# 07 — FX Channel Analysis
Oil→INR transmission: rolling β decomposition, dual-channel signals, regime detection.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('../src'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# fx_channel.py exact signatures:
#   fx_event_study(returns, events, pre, post, fx_col) -> pd.DataFrame
#   decompose_sector_returns(returns, stock_cols, oil_col, fx_col, window) -> Dict[str, pd.DataFrame]
#   fx_channel_summary(returns, events, stock_cols, oil_col, fx_col, window, post) -> pd.DataFrame
#   dual_channel_signal(returns, events, stock_cols, oil_col, fx_col, window, post) -> pd.DataFrame
#   oil_fx_regime(returns, oil_col, fx_col, window) -> pd.DataFrame
#   run_fx_analysis(returns, events, stock_cols, output_dir) -> Dict
from fx_channel import (
    fx_event_study, decompose_sector_returns, fx_channel_summary,
    dual_channel_signal, oil_fx_regime, run_fx_analysis, STOCK_COLS,
)
from event_study import OIL_SHOCK_EVENTS
from utils import DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, FX_COL, STOCK_UNIVERSE

# Convenience aliases used in save/plot calls below
TABLES = TABLES_DIR
PLOTS  = PLOTS_DIR


In [2]:
returns = pd.read_parquet(DATA_PROC / 'returns.parquet')
print(f'Data: {returns.shape}  {returns.index.min().date()} — {returns.index.max().date()}')
print(f'Columns: {list(returns.columns)}')


Data: (1928, 18)  2019-01-02 — 2026-05-25
Columns: ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA', 'BRENT', 'NIFTY', 'USDINR']


In [3]:
# USDINR event study — average INR response around oil shocks
# fx_event_study(returns, events, pre=5, post=5, fx_col='USDINR') -> pd.DataFrame
fx_study = fx_event_study(returns, OIL_SHOCK_EVENTS, pre=5, post=10)
print(fx_study)
fx_study.to_csv(TABLES / 'fx_usdinr_event_study.csv')


       mean     std  n_events  t_stat  p_value  cum_mean
day                                                     
-5  -0.1735  0.2759        10  -1.988   0.0781   -0.1735
-4  -0.0009  0.2734        10  -0.010   0.9921   -0.1744
-3   0.0357  0.2994        10   0.377   0.7148   -0.1387
-2   0.3115  0.2962        10   3.326   0.0089    0.1728
-1   0.0747  0.3435        10   0.688   0.5091    0.2475
0   -0.0231  0.2744        10  -0.266   0.7959    0.2244
1    0.0422  0.3091        10   0.432   0.6762    0.2666
2   -0.1241  0.3096        10  -1.268   0.2366    0.1425
3    0.1791  0.4752        10   1.192   0.2637    0.3216
4   -0.0285  0.3055        10  -0.295   0.7748    0.2931
5   -0.1005  0.3426        10  -0.928   0.3776    0.1926
6    0.0727  0.2788        10   0.825   0.4308    0.2653
7    0.0183  0.2929        10   0.198   0.8477    0.2836
8    0.0433  0.3169        10   0.432   0.6759    0.3269
9    0.0852  0.4403        10   0.612   0.5556    0.4121
10  -0.1868  0.2807        10  

In [4]:
set_theme()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ['#2E86AB' if v > 0 else '#E84855' for v in fx_study['mean']]
ax.bar(fx_study.index, fx_study['mean'], color=colors, alpha=0.8, width=0.8)
ax.axvline(0, color='k', lw=1.2, ls='--', alpha=0.6)
ax.set_xlabel('Days relative to oil shock')
ax.set_ylabel('Avg USDINR log return')
ax.set_title('Daily USDINR Response to Oil Shocks')

ax = axes[1]
ax.plot(fx_study.index, fx_study['cum_mean'], 'o-', color='#2E86AB', lw=2, ms=6)
ax.axvline(0, color='k', lw=1.2, ls='--', alpha=0.6)
ax.axhline(0, color='grey', lw=0.8, alpha=0.5)
ax.set_xlabel('Days relative to oil shock')
ax.set_ylabel('Cumulative USDINR log return')
ax.set_title('Cumulative INR Depreciation After Oil Shock')

plt.tight_layout()
plt.savefig(PLOTS / 'fx_usdinr_event_study.png', dpi=150, bbox_inches='tight')
plt.show()


In [5]:
# Rolling OLS decomposition: R_stock ~ β_oil·R_brent + β_fx·R_usdinr
# decompose_sector_returns(returns, stock_cols, oil_col, fx_col, window) -> Dict[str, pd.DataFrame]
betas = decompose_sector_returns(returns, STOCK_COLS, window=60)

summary_rows = []
for stock, df in betas.items():
    summary_rows.append({
        'stock':         stock,
        'mean_beta_oil': round(df['beta_oil'].mean(), 3),
        'mean_beta_fx':  round(df['beta_fx'].mean(),  3),
        'mean_r2':       round(df['r2'].mean(),        3),
    })

beta_summary = (pd.DataFrame(summary_rows)
                  .set_index('stock')
                  .sort_values('mean_beta_fx', ascending=False))
print(beta_summary)
beta_summary.to_csv(TABLES / 'fx_beta_summary.csv')


            mean_beta_oil  mean_beta_fx  mean_r2
stock                                           
BPCL                0.003         0.114    0.029
ONGC                0.001         0.105    0.040
CIPLA              -0.003         0.032    0.027
INDIGO             -0.046         0.020    0.044
INFY                0.006         0.014    0.038
IOC                -0.040        -0.001    0.039
MARUTI             -0.002        -0.003    0.029
HINDUNILVR          0.004        -0.008    0.036
TCS                 0.020        -0.038    0.036
TATAMOTORS          0.005        -0.043    0.027
ADANIPORTS          0.012        -0.051    0.037
ITC                -0.008        -0.057    0.030
ASIANPAINT         -0.021        -0.081    0.033
RELIANCE           -0.010        -0.094    0.042
HPCL                0.015        -0.119    0.034


In [6]:
set_theme()
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2E86AB' if v > 0 else '#E84855' for v in beta_summary['mean_beta_fx']]
ax.barh(beta_summary.index, beta_summary['mean_beta_fx'], color=colors, alpha=0.85)
ax.axvline(0, color='k', lw=1.2)
ax.set_xlabel('Mean β_fx  (rolling 60-day OLS)')
ax.set_title('FX Channel Sensitivity by Stock\nPositive = benefits from INR depreciation')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS / 'fx_beta_barh.png', dpi=150, bbox_inches='tight')
plt.show()


In [7]:
# FX channel summary per stock per event
fx_summary = fx_channel_summary(returns, OIL_SHOCK_EVENTS, STOCK_COLS)
print(fx_summary.to_string())
fx_summary.to_csv(TABLES / 'fx_channel_summary.csv')


            beta_oil  beta_fx  fx_car_pct prior_fx_dir  fx_confirmed
stock                                                               
BPCL           0.003    0.114      -0.004           ≈0         False
ONGC           0.001    0.105      -0.003            +          True
CIPLA         -0.003    0.032      -0.001            +         False
INDIGO        -0.046    0.020      -0.001            -         False
INFY           0.006    0.014      -0.000            +         False
IOC           -0.040   -0.001       0.000           ≈0         False
MARUTI        -0.002   -0.003       0.000            -         False
HINDUNILVR     0.004   -0.008       0.000            -         False
TCS            0.020   -0.038       0.001            +         False
TATAMOTORS     0.005   -0.043       0.001            -         False
ADANIPORTS     0.012   -0.051       0.002            +         False
ITC           -0.008   -0.057       0.002            -          True
ASIANPAINT    -0.021   -0.081     

In [8]:
# Dual-channel signal: oil + FX composite scores
signals = dual_channel_signal(returns, OIL_SHOCK_EVENTS, STOCK_COLS)
print(f'Signal rows: {len(signals)}')

avg_score = (signals.groupby('stock')['composite_score']
                    .mean()
                    .sort_values(ascending=False))
print('\nAverage composite score per stock:')
print(avg_score.round(3).to_string())
signals.to_csv(TABLES / 'fx_dual_channel_signals.csv', index=False)


Signal rows: 150

Average composite score per stock:
stock
IOC           0.654
BPCL          0.435
HPCL          0.425
TATAMOTORS    0.354
ASIANPAINT    0.291
TCS           0.286
INFY         -0.038
ONGC         -0.063
RELIANCE     -0.065
CIPLA        -0.068
HINDUNILVR   -0.284
ADANIPORTS   -0.344
MARUTI       -0.353
ITC          -0.432
INDIGO       -0.750


In [9]:
set_theme()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

stock_avg = signals.groupby('stock')[['beta_oil', 'beta_fx', 'composite_score']].mean()
ax = axes[0]
sc = ax.scatter(stock_avg['beta_oil'], stock_avg['beta_fx'],
                c=stock_avg['composite_score'],
                cmap='RdYlGn', s=120, alpha=0.85, edgecolors='k', lw=0.5)
for stock, row in stock_avg.iterrows():
    ax.annotate(stock, (row['beta_oil'], row['beta_fx']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.axhline(0, color='grey', lw=0.8, alpha=0.6)
ax.axvline(0, color='grey', lw=0.8, alpha=0.6)
ax.set_xlabel('β_oil'); ax.set_ylabel('β_fx')
ax.set_title('Dual-Channel Factor Map')
plt.colorbar(sc, ax=ax, label='composite score')

ax = axes[1]
colors = ['#2E86AB' if v > 0 else '#E84855' for v in avg_score]
ax.barh(avg_score.index, avg_score, color=colors, alpha=0.85)
ax.axvline(0, color='k', lw=1.2)
ax.set_xlabel('Average composite signal score')
ax.set_title('Dual-Channel Ranking Per Stock')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS / 'fx_dual_channel.png', dpi=150, bbox_inches='tight')
plt.show()


In [10]:
# Oil-FX coupling regime
regime = oil_fx_regime(returns)
print(regime['coupling_regime'].value_counts())

set_theme()
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(regime.index, regime['oil_fx_corr'], lw=1.2, color='#2E86AB')
ax.axhline( 0.4, color='g',      lw=1,   ls='--', alpha=0.7, label='tight (|ρ|>0.4)')
ax.axhline(-0.4, color='g',      lw=1,   ls='--', alpha=0.7)
ax.axhline( 0.2, color='orange', lw=0.8, ls=':',  alpha=0.6, label='moderate boundary')
ax.axhline(-0.2, color='orange', lw=0.8, ls=':',  alpha=0.6)
ax.axhline(0,    color='k',      lw=0.7, alpha=0.5)
ax.set_ylabel('Rolling 60-day ρ(Brent, USDINR)')
ax.set_title('Oil-FX Coupling Regime')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS / 'fx_coupling_regime.png', dpi=150, bbox_inches='tight')
plt.show()
print('NB07 complete ✓')


coupling_regime
loose       1605
moderate     204
tight         60
Name: count, dtype: int64
NB07 complete ✓
